## Introduction
This notebook trains several machine learning models to predict close price from a property's physical and geographic features and evaluates their performance on the validation set.

In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv('clean-data/sold_train.csv')
val = pd.read_csv('clean-data/sold_validation.csv')
test = pd.read_csv('clean-data/sold_test.csv')

## Part 1: Setup
#### 1.1 Load machine learning libraries and preprocessing pipeline

In [3]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from category_encoders import TargetEncoder
from utilities import impute_groupwise, get_preprocessor

In [4]:
from sklearn import set_config
import warnings

# suppress warnings from taking the mean of empty arrays
warnings.filterwarnings(action='ignore', message='Mean of empty slice')
# force every transformer to output clean pandas dataframes instead of raw numpy arrays
set_config(transform_output='pandas')

#### 1.2 Initiate performance tracker
Create a class to compute, log, and manage evaluation metrics across multiple machine learning models.

In [5]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [6]:
class ModelPerformanceTracker:
    # set tracker schema
    def __init__(self):
        self.summary = pd.DataFrame(columns=[
            'Model', 'Feature Size', 'R2', 'RMSE ($)', 'MAE ($)', 'MAPE (%)', 'MdAPE (%)'
        ])

    # calculate evaluation metrics and append them to the summary table
    def log_results(self, model_name, feature_size, y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)

        eps = 1e-8 # have an epsilon in place to prevent division by 0
        percentage_errors = np.abs((y_true - y_pred) / (y_true + eps)) * 100
        mape = np.mean(percentage_errors)
        mdape = np.median(percentage_errors)
        
        model_results = pd.DataFrame([{
            'Model': model_name,
            'Feature Size': feature_size,
            'R2': round(r2, 4),
            'RMSE ($)': round(rmse, 2),
            'MAE ($)': round(mae, 2), 
            'MAPE (%)': round(mape, 2),
            'MdAPE (%)': round(mdape, 2)
        }])
        if not model_results.empty:
            self.summary = pd.concat([self.summary, model_results], ignore_index=True)

    # return model performance summary sorted by MdAPE  
    def get_summary(self):
        return self.summary.sort_values(by='MdAPE (%)', ascending=True).reset_index(drop=True)

In [7]:
# initiate the tracker
tracker = ModelPerformanceTracker()

## Part 2: Modeling and Tuning
Train different machine learning models with a predefined feature set.

In [8]:
location_cols = ['MLSAreaMajor', 'CountyOrParish', 'City', 'PostalCode', 'DistrictNa']
bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']
numeric_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 
                'LotSizeSquareFeet', 'ParkingTotal', 'Stories', 
                'property_age']
X = location_cols + bool_cols + numeric_cols

In [9]:
X_train = train[X]
y_train = train['ClosePrice']

X_val = val[X]
y_val = val['ClosePrice']

X_test = test[X]
y_test = test['ClosePrice']

#### 2.1 Regression Models
Build linear regression models with a log-transformed target, beginning with ordinary least squares (OLS) and then extending to ridge regression and splines.

In [10]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import SplineTransformer

In [11]:
linear_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=True
)

In [12]:
ols_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', LinearRegression())
])

ols_model = TransformedTargetRegressor(
    regressor=ols_pipeline, func=np.log1p, inverse_func=np.expm1
)

ols_model.fit(X_train, y_train)
ols_pred_val = ols_model.predict(X_val)
tracker.log_results(
    model_name='Linear Regression', 
    feature_size=len(X),
    y_true=y_val, 
    y_pred=ols_pred_val
)

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_12174/1668477167.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.summary = pd.concat([self.summary, model_results], ignore_index=True)


In [13]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26


In [14]:
ridge_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', Ridge(alpha=1.0)) # L2 penalty
])

ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipeline, func=np.log1p, inverse_func=np.expm1
)

ridge_model.fit(X_train, y_train)
ridge_pred_val = ridge_model.predict(X_val)
tracker.log_results(
    model_name='Ridge Regression',
    feature_size=len(X),
    y_true=y_val,
    y_pred=ridge_pred_val
)

In [15]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
1,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


Given that regularization does not significantly improve the performance of linear regression models, try natural splines to introduce non-linearity.

In [16]:
n_knots = 5
degree = 3 # natural splines use cubic polynomials

In [17]:
spline_transformer = ColumnTransformer(
    transformers=[
        ('splines', 
         SplineTransformer(
             n_knots=n_knots, 
             degree=degree, 
             extrapolation='linear', # enforce linear boundaries
             include_bias=False),    # exclude intercept to reduce multi-collinearity
         make_column_selector(dtype_include=['float64', 'float32', 'int64', 'int32']))
    ],
    remainder='passthrough' 
)

spline_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('natural_splines', spline_transformer),
    ('estimator', LinearRegression())  
])

spline_model = TransformedTargetRegressor(
    regressor=spline_pipeline, func=np.log1p, inverse_func=np.expm1
)

In [18]:
spline_model.fit(X_train, y_train)
spline_pred_val = spline_model.predict(X_val)

tracker.log_results(
    model_name=f'Natural Spline ({n_knots} Knots)',
    feature_size=len(X),
    y_true=y_val,
    y_pred=spline_pred_val
)

In [19]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
1,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
2,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


In [20]:
n_knots = 8

In [21]:
spline_transformer = ColumnTransformer(
    transformers=[
        ('splines', 
         SplineTransformer(
             n_knots=n_knots, 
             degree=degree, 
             extrapolation='linear', # enforce linear boundaries
             include_bias=False),    # exclude intercept to reduce multi-collinearity
         make_column_selector(dtype_include=['float64', 'float32', 'int64', 'int32']))
    ],
    remainder='passthrough' 
)

spline_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('natural_splines', spline_transformer),
    ('estimator', LinearRegression())  
])

spline_model = TransformedTargetRegressor(
    regressor=spline_pipeline, func=np.log1p, inverse_func=np.expm1
)

In [22]:
spline_model.fit(X_train, y_train)
spline_pred_val = spline_model.predict(X_val)

tracker.log_results(
    model_name=f'Natural Spline ({n_knots} Knots)',
    feature_size=len(X),
    y_true=y_val,
    y_pred=spline_pred_val
)

In [23]:
# increasing the number of knots provides only marginal improvement in MdAPE, suggesting more advanced models are needed
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
1,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
2,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
3,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


#### 2.2 Tree-Based Models
Build tree-based models with a log-transformed target, beginning with decision tree and then extending to random forest and gradient boosting.

In [24]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [25]:
tree_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=False # no need to normalize numeric features for tree architectures
)

Decision trees:

In [26]:
depth = 12

In [27]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=dt_pred_val
)

In [28]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
1,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
2,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
3,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
4,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


In [29]:
depth = 10

In [30]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=dt_pred_val
)

In [31]:
# a shallower tree shows decline in performance, suggesting underfitting
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
1,Decision Tree (Depth=10),17,0.8376,376185.68,199862.31,14.94,10.79
2,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
3,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
4,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
5,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


In [32]:
depth = 14

In [33]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=dt_pred_val
)

In [34]:
# a slight increase in tree depth notably improves performance; try increasing it further
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Decision Tree (Depth=14),17,0.8233,392370.28,196032.65,14.34,9.68
1,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
2,Decision Tree (Depth=10),17,0.8376,376185.68,199862.31,14.94,10.79
3,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
4,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
5,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
6,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


In [35]:
depth = 16

In [36]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=dt_pred_val
)

In [37]:
# no change in MdAPE with increased tree depth, suggesting limited benefit from further increases
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Decision Tree (Depth=14),17,0.8233,392370.28,196032.65,14.34,9.68
1,Decision Tree (Depth=16),17,0.8151,401319.98,198138.30,14.52,9.68
2,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
3,Decision Tree (Depth=10),17,0.8376,376185.68,199862.31,14.94,10.79
4,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
5,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
6,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
7,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


Random forests:

In [38]:
n_trees = 150
depth = 14 # based on the decision tree evaluations earlier

In [39]:
rf_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=42, n_jobs=-1))
])

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
)

rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
tracker.log_results(
    model_name=f'Random Forest ({n_trees} Trees, Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=rf_pred_val
)

In [40]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (150 Trees, Depth=14)",17,0.8768,327596.30,169890.45,12.39,8.78
1,Decision Tree (Depth=14),17,0.8233,392370.28,196032.65,14.34,9.68
2,Decision Tree (Depth=16),17,0.8151,401319.98,198138.30,14.52,9.68
3,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
4,Decision Tree (Depth=10),17,0.8376,376185.68,199862.31,14.94,10.79
5,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
6,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
7,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
8,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


In [41]:
n_trees = 180

In [42]:
rf_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=42, n_jobs=-1))
])

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
)

rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
tracker.log_results(
    model_name=f'Random Forest ({n_trees} Trees, Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=rf_pred_val
)

In [43]:
# increasing the number of trees yields marginal improvement in performance, suggesting more advanced models are needed
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (180 Trees, Depth=14)",17,0.8768,327651.16,169913.19,12.39,8.76
1,"Random Forest (150 Trees, Depth=14)",17,0.8768,327596.30,169890.45,12.39,8.78
2,Decision Tree (Depth=14),17,0.8233,392370.28,196032.65,14.34,9.68
3,Decision Tree (Depth=16),17,0.8151,401319.98,198138.30,14.52,9.68
4,Decision Tree (Depth=12),17,0.8344,379882.61,195946.92,14.53,10.17
5,Decision Tree (Depth=10),17,0.8376,376185.68,199862.31,14.94,10.79
6,Natural Spline (8 Knots),17,0.8396,373844.26,201099.96,15.10,11.28
7,Natural Spline (5 Knots),17,0.8374,376424.57,202725.18,15.29,11.34
8,Linear Regression,17,0.8009,416528.57,217195.65,16.35,12.26
9,Ridge Regression,17,0.8009,416531.50,217196.40,16.35,12.26


Gradient boosting:

Both random forest and gradient boosting are ensembles of decision trees, but gradient boosting builds trees sequentially, with each tree targeting the errors of the previous trees, making it more powerful in many cases. Therefore, we next try gradient boosting, starting with XGBoost, which controls complexity level-wise (i.e. tree depth), followed by LightGBM, which controls complexity leaf-wise (i.e. number of leaves).

In [44]:
%pip install xgboost lightgbm

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
